# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule:** rank pages by low volume. The fewer h1 impressions a page has, the higher it ranks, because low-volume pages declined most often in my check. Inputs are 1-15 March only.

**Reason codes:** LOW_VOLUME_DECAY (top 10% of the ranking) and NO_FLAG (everything else).
**Action labels:** refresh_review for LOW_VOLUME_DECAY, monitor for NO_FLAG.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.execute(f"CREATE VIEW mar AS SELECT * FROM read_parquet('{MAR}')")

frame = con.sql("""
SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS avg_position_h1,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS active_days_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '2026-03-15') AS clicks_h2
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df = frame.copy()
df["clicks_h2"] = df["clicks_h2"].fillna(0)
df["ctr_h1"] = df["clicks_h1"] / df["impressions_h1"].replace(0, np.nan)
FEATURES = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
df = df[df["clicks_h1"] >= 5].dropna(subset=FEATURES).copy()
df["label"] = (df["clicks_h2"] < 0.8 * df["clicks_h1"]).astype(int)   # sirf verdict ke liye

print("rows:", len(df), "| overall decline rate:", round(df["label"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 17431 | overall decline rate: 0.442


In [13]:
def bucket_table(d, col, q=5):
    b = pd.qcut(d[col], q=q, duplicates="drop")
    t = (d.assign(bucket=b).groupby("bucket", observed=True)
           .agg(n=("label", "size"), decline_rate=("label", "mean"),
                median_ctr=("ctr_h1", "median"))
           .round(3))
    print(f"--- {col} ---")
    print(t, "\n")
    return t

# Signal A: volume
tA = bucket_table(df, "impressions_h1")

# Signal B: CTR apni position ke hisaab se kitna kam/zyada
df["pos_band"] = pd.cut(df["avg_position_h1"], [0, 3, 10, 20, 1000], labels=["1-3", "4-10", "11-20", "20+"])
df["ctr_vs_pos"] = df["ctr_h1"] / df.groupby("pos_band", observed=True)["ctr_h1"].transform("median")
tB = bucket_table(df, "ctr_vs_pos")

--- impressions_h1 ---
                       n  decline_rate  median_ctr
bucket                                            
(6.999, 1172.0]     3488         0.509       0.010
(1172.0, 2054.0]    3487         0.470       0.005
(2054.0, 3363.0]    3485         0.430       0.004
(3363.0, 6152.0]    3485         0.412       0.003
(6152.0, 161575.0]  3486         0.390       0.002 

--- ctr_vs_pos ---
                                  n  decline_rate  median_ctr
bucket                                                       
(0.033299999999999996, 0.505]  3487         0.414       0.002
(0.505, 0.823]                 3486         0.449       0.003
(0.823, 1.218]                 3486         0.436       0.005
(1.218, 1.933]                 3486         0.452       0.007
(1.933, 151.89]                3486         0.460       0.013 



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = df.copy()
q["impr_pct"] = q["impressions_h1"].rank(pct=True)
q["score"] = (1 - q["impr_pct"]).round(4)

cut = q["score"].quantile(0.90)
q["action_label"] = np.where(q["score"] >= cut, "refresh_review", "monitor")
q["reason_code"] = np.where(q["score"] >= cut, "LOW_VOLUME_DECAY", "NO_FLAG")

q = q.sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
print("rows:", len(q))

rows: 17431


In [15]:
import os
os.makedirs("work/outputs", exist_ok=True)

OUT = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action_label",
       "impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
q[OUT].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("written:", len(q), "rows")
q[OUT].head(10)

written: 17431 rows


,rank,client_hash_id,content_hash_id,score,reason_code,action_label,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,1,client_3ffa76342f366962,content_398d2c4de4555c8d,0.9999,LOW_VOLUME_DECAY,refresh_review,7.0,5.0,0.714286,3.583333,6
1,2,client_f623b01661d4bfe4,content_4663d6efb6744b2d,0.9999,LOW_VOLUME_DECAY,refresh_review,32.0,5.0,0.156250,7.486111,12
2,3,client_08a6a72ff48e62c0,content_f25fdf890ef2b482,0.9999,LOW_VOLUME_DECAY,refresh_review,32.0,6.0,0.187500,7.237500,4
3,4,client_3ffa76342f366962,content_76aefc7358625599,0.9998,LOW_VOLUME_DECAY,refresh_review,44.0,5.0,0.113636,3.464646,11
4,5,client_3ffa76342f366962,content_924fcfa2714a8454,0.9997,LOW_VOLUME_DECAY,refresh_review,46.0,7.0,0.152174,2.370238,14
5,6,client_2094c6eb080311d5,content_6546112a234cb98d,0.9997,LOW_VOLUME_DECAY,refresh_review,57.0,5.0,0.087719,4.365079,6
6,7,client_23a62021009f63c4,content_7d87c69eaba32c8c,0.9996,LOW_VOLUME_DECAY,refresh_review,60.0,6.0,0.100000,5.716392,13
7,8,client_23a62021009f63c4,content_c0a6447bd5760c74,0.9995,LOW_VOLUME_DECAY,refresh_review,67.0,5.0,0.074627,3.532271,4
8,9,client_3ffa76342f366962,content_b4f3e4b02ea7dc72,0.9995,LOW_VOLUME_DECAY,refresh_review,62.0,5.0,0.080645,9.996481,15
9,10,client_2094c6eb080311d5,content_c4d371ac4ef2a8ba,0.9994,LOW_VOLUME_DECAY,refresh_review,69.0,5.0,0.072464,2.374579,11


## 3. Top-20 review

**Top-20 review.** All twenty are refresh_review with reason code LOW_VOLUME_DECAY, because the rule ranks by low h1 impressions only. The table above gives, for each row, the action, the reason code, a confidence note, and what would make it wrong. Observed on March 2026 data, directional only.

**Overall confidence: low.** 16 of the 20 rows have only 5 to 7 clicks, so one or two clicks can flip the decline label. Only 4 rows (ranks 13, 15, 18, 20) have 8 or more clicks and get a medium note.

**Client concentration:** the 20 rows come from 9 clients, but one client has 6 rows and the top three clients have 13 of the 20. The queue may reflect those clients more than the whole panel.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top = q.head(20).copy()

def wrong(r):
    notes = []
    if r.clicks_h1 <= 7:
        notes.append(f"only {int(r.clicks_h1)} clicks, a 1-2 click change flips the label")
    if r.active_days_h1 <= 5:
        notes.append(f"active only {int(r.active_days_h1)} days, h1 may not be typical")
    if r.avg_position_h1 <= 5:
        notes.append("good position, may be a stable niche page")
    if r.avg_position_h1 >= 9:
        notes.append("position 9 or worse, rank wobble may drive the change")
    return "; ".join(notes) or "normal h2 traffic would simply hold up"

top["confidence"] = np.where(top["clicks_h1"] <= 7, "low", "medium")
top["what_would_make_it_wrong"] = top.apply(wrong, axis=1)

print("clients in top 20:")
print(top["client_hash_id"].value_counts(), "\n")

pd.set_option("display.max_colwidth", 140)
top[["rank", "action_label", "reason_code", "confidence",
     "what_would_make_it_wrong", "impressions_h1", "clicks_h1",
     "avg_position_h1", "active_days_h1"]]


clients in top 20:
client_hash_id
client_3ffa76342f366962    6
client_2094c6eb080311d5    3
client_23a62021009f63c4    3
client_08a6a72ff48e62c0    2
client_f623b01661d4bfe4    1
client_d211cb07b9059bab    1
client_fef1a8f436438636    1
client_73cda7b4e4f265ea    1
client_a80fca3f171ed1de    1
client_20259bd6705d81d4    1
Name: count, dtype: int64 



,rank,action_label,reason_code,confidence,what_would_make_it_wrong,impressions_h1,clicks_h1,avg_position_h1,active_days_h1
0,1,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; good position, may be a stable niche page",7.0,5.0,3.583333,6
1,2,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label",32.0,5.0,7.486111,12
2,3,refresh_review,LOW_VOLUME_DECAY,low,"only 6 clicks, a 1-2 click change flips the label; active only 4 days, h1 may not be typical",32.0,6.0,7.237500,4
3,4,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; good position, may be a stable niche page",44.0,5.0,3.464646,11
4,5,refresh_review,LOW_VOLUME_DECAY,low,"only 7 clicks, a 1-2 click change flips the label; good position, may be a stable niche page",46.0,7.0,2.370238,14
5,6,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; good position, may be a stable niche page",57.0,5.0,4.365079,6
6,7,refresh_review,LOW_VOLUME_DECAY,low,"only 6 clicks, a 1-2 click change flips the label",60.0,6.0,5.716392,13
7,8,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; active only 4 days, h1 may not be typical; good position, may be a stable niche page",67.0,5.0,3.532271,4
8,9,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; position 9 or worse, rank wobble may drive the change",62.0,5.0,9.996481,15
9,10,refresh_review,LOW_VOLUME_DECAY,low,"only 5 clicks, a 1-2 click change flips the label; good position, may be a stable niche page",69.0,5.0,2.374579,11


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Result (measured, directional).** Among the 10% of pages flagged refresh_review, the decline rate is 0.53, against 0.442 for all rows. This is a modest lift, and it also means about 47% of flagged pages did not decline.

**Weak picks.**
- The flagged pages have a median of 6 clicks in 1-15 March, against 10 for all rows. With so few clicks, one or two clicks can flip the decline label, so part of the 0.53 may be noise. Some of it may also be regression to the mean: pages that happened to sit just above the 5-click floor tend to fall back.
- Several top-20 rows were active only 4 days in h1, so their first half may not be typical.
- The rule uses volume only. It does not know why a page declined (content age, seasonality, ranking change), so it says where to look, not what to fix.
- Several rows come from the same few clients, so the queue may reflect those clients more than the whole panel.
- I chose the rule after looking at the same month's decline label, so this is not a validated result. The Week-5 model must beat it on data it has not seen.

**Leakage check.** The score uses only impressions_h1, which comes from 1-15 March. The label (built from 16-31 March clicks) is used only to check the signals and the final result, never to build the score. The CSV contains no label or clicks_h2 column (checked: empty set), and no ga4_*, ai_*, or other product-flag columns were used. June 2026 was not touched.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flagged = q[q["action_label"] == "refresh_review"]
print("refresh_review decline rate:", round(flagged["label"].mean(), 3), "| all rows:", round(q["label"].mean(), 3))
print("flagged median clicks_h1:", flagged["clicks_h1"].median(), "| all median:", q["clicks_h1"].median())
print("label/future columns in CSV:", {"label", "clicks_h2"} & set(OUT))


refresh_review decline rate: 0.53 | all rows: 0.442
flagged median clicks_h1: 6.0 | all median: 10.0
label/future columns in CSV: set()


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.